# 02 — Preprocesamiento de datos

**Materia:** Inteligencia Artificial

**Proyecto:** Clasificación del uso de gas en garrafa en los hogares de Bolivia.

**Fuente de datos:** Encuesta de Hogares 2023 — INE Bolivia.

**Responsable:** Integrante 2 — Preprocesamiento y PCA.

## Objetivo

Realizar la limpieza, selección, codificación y estandarización de los datos para preparar un conjunto de variables adecuado para la aplicación de PCA, algoritmos de clasificación y técnicas de agrupamiento.


In [ ]:
# 1. Importación de librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

print("Librerías importadas correctamente")

In [ ]:
# 2. Conexión con Google Drive

from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# 3. Carga del dataset original

ruta = '/content/drive/MyDrive/PROYECTO_DE_IA/'

dataset = pd.read_csv(
    ruta + 'data/raw/EH2023_Vivienda.csv'
)

dataset.head()

In [ ]:
# 4. Dimensiones del dataset

print("Cantidad de filas:", dataset.shape[0])
print("Cantidad de columnas:", dataset.shape[1])


## Exploración y diagnóstico de calidad de los datos

En esta etapa se realiza una revisión de la estructura del conjunto de datos, identificando los tipos de variables, valores faltantes y registros duplicados. Este procedimiento permite detectar posibles inconsistencias antes de aplicar las técnicas de preprocesamiento y reducción de dimensionalidad.

In [ ]:
# 5. Información general del dataset

dataset.info()

In [ ]:
# 6. Identificación de valores faltantes

valores_faltantes = pd.DataFrame({
    'Cantidad': dataset.isnull().sum(),
    'Porcentaje (%)': (
        dataset.isnull().mean() * 100
    ).round(2)
})

valores_faltantes.sort_values(
    by='Cantidad',
    ascending=False
)

In [ ]:
# 7. Identificación de registros duplicados

duplicados = dataset.duplicated().sum()

print("Total de registros:", len(dataset))
print("Registros duplicados:", duplicados)
print("Porcentaje de duplicados:",
      round(duplicados / len(dataset) * 100, 2), "%")

In [ ]:
# 8. Exploración de la variable objetivo

print("Valores de la variable objetivo:")

display(
    dataset['s06a_15'].value_counts(dropna=False)
)

print("\nCantidad de valores faltantes:")

print(dataset['s06a_15'].isnull().sum())

## Selección de variables predictoras y variable objetivo

Después de realizar el diagnóstico de calidad de los datos, se identificaron 12.815 registros y 37 columnas. No se encontraron filas completamente duplicadas ni valores faltantes en la variable objetivo.

Se seleccionarán 12 variables predictoras relacionadas con las características geográficas, demográficas y habitacionales de los hogares.

La variable objetivo corresponde al combustible principal utilizado para cocinar (s06a_15), la cual será transformada en una variable binaria:

- 1: El hogar utiliza gas en garrafa como combustible principal.
- 0: El hogar no utiliza gas en garrafa como combustible principal.

Esta transformación permitirá aplicar posteriormente algoritmos de clasificación supervisada.

In [ ]:
# 9. Selección inicial de variables predictoras

variables_predictoras = [
    'depto',
    'area',
    'totper',
    's06a_01',
    's06a_02',
    's06a_03',
    's06a_04',
    's06a_05',
    's06a_06',
    's06a_07',
    's06a_12',
    's06a_16'
]

variable_objetivo = 's06a_15'

print("Cantidad de variables predictoras:",
      len(variables_predictoras))

print("Variable objetivo:", variable_objetivo)

display(dataset[variables_predictoras].head())

In [ ]:
# 10. Verificación de las variables predictoras

resumen_variables = pd.DataFrame({
    'Tipo de dato':
        dataset[variables_predictoras].dtypes,

    'Valores faltantes':
        dataset[variables_predictoras].isnull().sum(),

    'Valores únicos':
        dataset[variables_predictoras].nunique()
})

display(resumen_variables)

In [ ]:
# 11. Verificación de categorías de s06a_05

print("Categorías de s06a_05:")

display(
    dataset['s06a_05'].value_counts(dropna=False)
)

print("\nCategorías de s06a_09:")

display(
    dataset['s06a_09'].value_counts(dropna=False)
)

In [ ]:
# 12. Verificación de identificadores

print("Total de registros:", len(dataset))

print("Folios únicos:",
      dataset['folio'].nunique())

print("Folios duplicados:",
      dataset['folio'].duplicated().sum())

In [ ]:
# 13. Construcción de la variable objetivo

# Verificar códigos válidos
codigos_validos = [1, 2, 3, 4, 5, 6, 7]

assert dataset[variable_objetivo].isin(
    codigos_validos
).all(), "Existen códigos no válidos"

# Crear la variable objetivo binaria
Y = (dataset[variable_objetivo] == 3).astype(int)

Y.name = 'usa_garrafa'

# Mostrar distribución
print("Distribución de la variable objetivo:")

display(Y.value_counts().sort_index())

print("\nDistribución porcentual:")

display(
    (Y.value_counts(normalize=True)
      .sort_index() * 100).round(2)
)

In [ ]:
# 14. Extracción de códigos de la variable s06a_05

codigos_techo = (
    dataset['s06a_05']
    .astype(str)
    .str.extract(r'^\s*(\d+)')[0]
    .astype(int)
)

print("Códigos encontrados:")
display(codigos_techo.value_counts().sort_index())

print("\nCódigos válidos:",
      codigos_techo.isin([1, 2, 3, 4, 5]).all())

In [ ]:
# 15. Validación de códigos con el diccionario oficial

verificacion = pd.DataFrame({
    'codigo': codigos_techo,
    'factor': dataset['factor']
})

resumen_techo = verificacion.groupby('codigo').agg(
    registros=('codigo', 'size'),
    casos_ponderados=('factor', 'sum')
)

resumen_techo['porcentaje_ponderado'] = (
    resumen_techo['casos_ponderados'] /
    resumen_techo['casos_ponderados'].sum() * 100
).round(2)

display(resumen_techo)

## Corrección de las etiquetas de la variable s06a_05

Durante la exploración de los datos se identificó una inconsistencia en las etiquetas de la variable correspondiente al material del techo de la vivienda.

Se extrajeron los códigos numéricos y se compararon sus frecuencias ponderadas con las publicadas en el diccionario oficial de la Encuesta de Hogares 2023 del INE.

Los resultados coincidieron con la distribución oficial, por lo que se decidió conservar los códigos numéricos y corregir su interpretación mediante el diccionario oficial de categorías.

Esta corrección permite mantener las 12 variables predictoras seleccionadas inicialmente, sin eliminar registros ni modificar el archivo original.

In [ ]:
# 16. Corrección de la variable s06a_05

# Diccionario oficial de categorías
diccionario_techo = {
    1: 'Calamina o plancha',
    2: 'Teja (cemento, arcilla, fibrocemento)',
    3: 'Losa de hormigón armado',
    4: 'Paja, caña, palma o barro',
    5: 'Otro'
}

# Crear una copia del dataset original
dataset_limpio = dataset.copy()

# Conservar los códigos numéricos correctos
dataset_limpio['s06a_05'] = codigos_techo

# Mostrar las categorías corregidas
print("Categorías corregidas de s06a_05:")

display(
    dataset_limpio['s06a_05']
    .map(diccionario_techo)
    .value_counts()
)

print("\nTipo de dato corregido:",
      dataset_limpio['s06a_05'].dtype)

print("Valores faltantes:",
      dataset_limpio['s06a_05'].isnull().sum())

print("Total de registros:",
      len(dataset_limpio))

## Codificación y estandarización de las variables

Una vez realizada la limpieza y corrección de los datos, se mantienen las 12 variables predictoras seleccionadas inicialmente.

Se utilizará One-Hot Encoding para transformar las variables categóricas y StandardScaler para estandarizar las variables numéricas.

Previamente, se dividirá el conjunto de datos en entrenamiento y prueba para evitar fugas de información durante el ajuste de las transformaciones.

In [ ]:
# 17. Construcción definitiva de X e Y

variables_predictoras = [
    'depto',
    'area',
    'totper',
    's06a_01',
    's06a_02',
    's06a_03',
    's06a_04',
    's06a_05',
    's06a_06',
    's06a_07',
    's06a_12',
    's06a_16'
]

X = dataset_limpio[variables_predictoras].copy()

Y = (dataset_limpio['s06a_15'] == 3).astype(int)

Y.name = 'usa_garrafa'

print("Dimensiones de X:", X.shape)
print("Dimensiones de Y:", Y.shape)
print("Valores faltantes en X:", X.isnull().sum().sum())

display(X.head())

In [ ]:
# 18. Separación de variables numéricas y categóricas

variables_numericas = [
    'totper',
    's06a_16'
]

variables_categoricas = [
    variable
    for variable in variables_predictoras
    if variable not in variables_numericas
]

print("Cantidad de variables numéricas:",
      len(variables_numericas))

print(variables_numericas)

print("\nCantidad de variables categóricas:",
      len(variables_categoricas))

print(variables_categoricas)

In [ ]:
# 19. División de datos en entrenamiento y prueba

X_train, X_test, Y_train, Y_test = train_test_split(
    X,
    Y,
    test_size=0.20,
    random_state=42,
    stratify=Y
)

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de X_test:", X_test.shape)

print("\nDistribución de Y_train:")
display(Y_train.value_counts().sort_index())

print("\nDistribución de Y_test:")
display(Y_test.value_counts().sort_index())

In [ ]:
# 20. Creación del preprocesador

preprocesador = ColumnTransformer(
    transformers=[
        (
            'numericas',
            StandardScaler(),
            variables_numericas
        ),
        (
            'categoricas',
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False
            ),
            variables_categoricas
        )
    ],
    remainder='drop'
)

print("Preprocesador creado correctamente")

In [ ]:
# 21. Aplicación del preprocesamiento

X_train_procesado = preprocesador.fit_transform(
    X_train
)

X_test_procesado = preprocesador.transform(
    X_test
)

print("Entrenamiento antes:", X_train.shape)
print("Entrenamiento después:", X_train_procesado.shape)

print("\nPrueba antes:", X_test.shape)
print("Prueba después:", X_test_procesado.shape)

In [ ]:
# 22. Verificación de los datos transformados

nombres_columnas = preprocesador.get_feature_names_out()

X_train_df = pd.DataFrame(
    X_train_procesado,
    columns=nombres_columnas,
    index=X_train.index
)

X_test_df = pd.DataFrame(
    X_test_procesado,
    columns=nombres_columnas,
    index=X_test.index
)

print("Variables originales:", X.shape[1])

print("Variables transformadas:", X_train_df.shape[1])

print("\nValores faltantes en entrenamiento:",
      X_train_df.isnull().sum().sum())

print("Valores faltantes en prueba:",
      X_test_df.isnull().sum().sum())

display(X_train_df.head())

## Validación y exportación de los datos preprocesados

Como resultado de la aplicación de One-Hot Encoding y StandardScaler, las 12 variables predictoras originales se transformaron en 62 características numéricas.

Se obtuvieron 60 columnas binarias correspondientes a las variables categóricas y dos columnas correspondientes a las variables numéricas estandarizadas.

Se verificó que los conjuntos de entrenamiento y prueba no presentan valores faltantes. Posteriormente, se comprobarán las propiedades estadísticas de la estandarización y se exportará el conjunto de datos limpio para su utilización en los experimentos de PCA, clasificación y agrupamiento.

In [ ]:
# 23. Verificación de la estandarización

columnas_estandarizadas = [
    'numericas__totper',
    'numericas__s06a_16'
]

verificacion_estandarizacion = pd.DataFrame({
    'Media': X_train_df[
        columnas_estandarizadas
    ].mean(),

    'Desviación estándar': X_train_df[
        columnas_estandarizadas
    ].std(ddof=0)
})

display(verificacion_estandarizacion.round(6))

In [ ]:
# 24. Verificación de la codificación One-Hot

columnas_onehot = [
    columna
    for columna in X_train_df.columns
    if columna.startswith('categoricas__')
]

valores_unicos = np.unique(
    X_train_df[columnas_onehot].to_numpy()
)

print("Cantidad de columnas One-Hot:",
      len(columnas_onehot))

print("Valores encontrados:", valores_unicos)

print("Codificación binaria correcta:",
      np.isin(valores_unicos, [0, 1]).all())

In [ ]:
# 25. Validación general del preprocesamiento

assert X.shape == (12815, 12)

assert len(X) == len(Y)

assert X.index.equals(Y.index)

assert X_train_df.index.equals(Y_train.index)

assert X_test_df.index.equals(Y_test.index)

assert X_train_df.shape[1] == X_test_df.shape[1]

assert np.isfinite(X_train_procesado).all()

assert np.isfinite(X_test_procesado).all()

assert len(
    X_train.index.intersection(X_test.index)
) == 0

print("Todas las validaciones fueron exitosas")

print("\nEntrenamiento:", X_train_df.shape)

print("Prueba:", X_test_df.shape)

print("Total de registros:",
      len(X_train_df) + len(X_test_df))

In [ ]:
# 26. Exportación del dataset limpio

import os

# Unir las variables predictoras con la variable objetivo
datos_finales = X.copy()

datos_finales['usa_garrafa'] = Y

# Crear la carpeta de salida si no existe
os.makedirs(
    ruta + 'data/processed/',
    exist_ok=True
)

# Guardar el archivo CSV
ruta_salida = (
    ruta + 'data/processed/datos_limpios.csv'
)

datos_finales.to_csv(
    ruta_salida,
    index=False
)

print("Archivo guardado correctamente")

print("Ubicación:", ruta_salida)

print("Dimensiones:", datos_finales.shape)

display(datos_finales.head())

In [ ]:
# 27. Verificación del archivo exportado

datos_verificados = pd.read_csv(ruta_salida)

print("Dimensiones del archivo:",
      datos_verificados.shape)

print("Valores faltantes:",
      datos_verificados.isnull().sum().sum())

print("Registros duplicados según todas las columnas:",
      datos_verificados.duplicated().sum())

print("\nDistribución de la variable objetivo:")

display(
    datos_verificados['usa_garrafa']
    .value_counts()
    .sort_index()
)

assert datos_verificados.equals(
    datos_finales.reset_index(drop=True)
)

print("\nEl archivo coincide con los datos exportados")

## 7 Modularización del preprocesamiento

Con el propósito de garantizar la reproducibilidad de los experimentos y facilitar la integración del trabajo de los integrantes del equipo, se implementará un módulo independiente denominado preprocessing.py.

Este módulo permitirá cargar y validar los datos limpios, construir las variables predictoras y la variable objetivo, aplicar las transformaciones correspondientes y generar pipelines que incorporen opcionalmente PCA y algoritmos de clasificación.

El preprocesamiento y PCA deberán ajustarse exclusivamente con los datos de entrenamiento en cada partición de los experimentos de clasificación.

In [ ]:
# 28. Creación del módulo preprocessing.py

from pathlib import Path

codigo_preprocesamiento = '''
import pandas as pd

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA


# Variables predictoras originales

VARIABLES_PREDICTORAS = [
    'depto',
    'area',
    'totper',
    's06a_01',
    's06a_02',
    's06a_03',
    's06a_04',
    's06a_05',
    's06a_06',
    's06a_07',
    's06a_12',
    's06a_16'
]

VARIABLE_OBJETIVO = 'usa_garrafa'

VARIABLES_NUMERICAS = [
    'totper',
    's06a_16'
]

VARIABLES_CATEGORICAS = [
    variable
    for variable in VARIABLES_PREDICTORAS
    if variable not in VARIABLES_NUMERICAS
]


# Función para cargar los datos limpios

def cargar_datos(ruta_csv):

    datos = pd.read_csv(ruta_csv)

    X = datos[VARIABLES_PREDICTORAS].copy()

    Y = datos[VARIABLE_OBJETIVO].copy()

    if X.isnull().any().any() or Y.isnull().any():
        raise ValueError("Existen valores faltantes")

    if not Y.isin([0, 1]).all():
        raise ValueError("Variable objetivo no válida")

    return X, Y


# Función para crear el preprocesador

def crear_preprocesador():

    return ColumnTransformer(
        transformers=[
            (
                'numericas',
                StandardScaler(),
                VARIABLES_NUMERICAS
            ),
            (
                'categoricas',
                OneHotEncoder(
                    handle_unknown='ignore',
                    sparse_output=False
                ),
                VARIABLES_CATEGORICAS
            )
        ],
        remainder='drop'
    )


# Función para crear un pipeline

def crear_pipeline(n_componentes=None, modelo=None):

    pasos = [
        ('preprocesamiento', crear_preprocesador())
    ]

    if n_componentes is not None:

        pasos.append(
            ('pca', PCA(
                n_components=n_componentes,
                svd_solver='full'
            ))
        )

    if modelo is not None:

        pasos.append(
            ('modelo', modelo)
        )

    return Pipeline(pasos)
'''

# Crear la carpeta src si no existe

carpeta_src = Path(ruta) / 'src'

carpeta_src.mkdir(
    parents=True,
    exist_ok=True
)

# Guardar el módulo

archivo_python = carpeta_src / 'preprocessing.py'

archivo_python.write_text(
    codigo_preprocesamiento,
    encoding='utf-8'
)

print("Módulo creado correctamente")

print("Ubicación:", archivo_python)

In [ ]:
# 29. Importación y prueba del módulo

import sys
import importlib

if str(carpeta_src) not in sys.path:
    sys.path.insert(0, str(carpeta_src))

import preprocessing

importlib.reload(preprocessing)

# Cargar los datos mediante nuestro módulo

X_modulo, Y_modulo = preprocessing.cargar_datos(
    ruta_salida
)

print("Dimensiones de X:", X_modulo.shape)

print("Dimensiones de Y:", Y_modulo.shape)

# Comparar con los datos anteriores

assert X_modulo.equals(X)

assert Y_modulo.equals(Y)

print("\nEl módulo funciona correctamente")

In [ ]:
# 30. Verificación del módulo de preprocesamiento

preprocesador_modulo = (
    preprocessing.crear_preprocesador()
)

X_train_modulo = (
    preprocesador_modulo.fit_transform(X_train)
)

X_test_modulo = (
    preprocesador_modulo.transform(X_test)
)

print("Entrenamiento:", X_train_modulo.shape)

print("Prueba:", X_test_modulo.shape)

# Comparar los resultados con los anteriores

assert np.allclose(
    X_train_modulo,
    X_train_procesado
)

assert np.allclose(
    X_test_modulo,
    X_test_procesado
)

print("\nLos resultados coinciden correctamente")